# 21 · FoodNExTDB calibration and annotation-agreement selection

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

This gives an immediately executable public-data research baseline after notebooks 00–05. It is not the flagship cancer-relevant claim-support experiment.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Calibrate foundation model probabilities

In [ ]:
from oncoplate.pipeline import calibrate_predictor,partition_predictions,prediction_arrays,load_study
from oncoplate.calibration import probabilities,calibration_metrics
from oncoplate.selection import choose_threshold
import numpy as np,pandas as pd
assert cfg['study']['dataset']=='foodnextdb','Foundation notebook has its own dataset and estimand.'
RUN_IDS=['resnet50_frozen_independent_s0','resnet50_frozen_joint_s0']
for run_id in RUN_IDS:
    cals,metrics=calibrate_predictor(cfg,run_id)
    print(run_id,{k:v['log_loss'] for k,v in metrics.items()})

## 2. Lock an annotation-agreement selection threshold on validation

In [ ]:
from oncoplate.statistics import analysis_weights
from oncoplate.governance import make_lock
thresholds={}
for run_id in RUN_IDS:
    run=p['runs']/run_id;pred=prediction_arrays(run/'validation_predictions.npz')
    cal=read_json(run/'calibrators.json')['temperature'];pr=probabilities(pred['logits'],cal)
    records,_=load_study(cfg,read_json(run/'run.json')['spec']['head'])
    sub=records.set_index('record_id').loc[pred['ids']].reset_index()
    thresholds[run_id]=choose_threshold(pr.max(1),np.ones(len(pr),bool),.8,analysis_weights(sub,{'meal':1.}))
write_json(p['private']/'foundation_thresholds.json',thresholds)
files=[p['prepared']/'split_manifest.csv',p['private']/'foundation_thresholds.json']
for run_id in RUN_IDS:files += [p['runs']/run_id/'best.pt',p['runs']/run_id/'calibrators.json']
lock_path=p['private']/'analysis_lock.json'
if not lock_path.exists():make_lock(lock_path,files,{'scope':'public_expert_visual_annotation_agreement_only','run_ids':RUN_IDS,'target_coverage':.8})
print(thresholds)

## 3. Evaluate the locked public-data baseline

In [ ]:
from oncoplate.statistics import foundation_group_intervals
rows=[]
for run_id in RUN_IDS:
    pred=partition_predictions(cfg,run_id,'test',allow_unblind=True)
    cal=read_json(p['runs']/run_id/'calibrators.json')['temperature'];pr=probabilities(pred['logits'],cal)
    j=pr.argmax(1);confidence=pr.max(1);accepted=confidence>=thresholds[run_id]['threshold']
    observed=pred['mask'][np.arange(len(pr)),j]>0
    agreement=pred['y'][np.arange(len(pr)),j]
    records,_=load_study(cfg,read_json(p['runs']/run_id/'run.json')['spec']['head'])
    sub=records.set_index('record_id').loc[pred['ids']].reset_index();w=analysis_weights(sub,{'meal':1.})
    assessed=accepted&observed;den=w[assessed].sum()
    rows.append({'run_id':run_id,'answer_coverage_all_records':float(w[accepted].sum()),'assessable_answer_coverage':float(den),
      'panel_disagreement_among_assessable_answers':float(np.sum(w[assessed]*(1-agreement[assessed]))/den) if den else np.nan,
      'unassessable_accepted_records':int((accepted&~observed).sum()),'reference':'expert_visual_panel_endorsement_not_documented_preparation'})
    frame=sub[['record_id','group_id']].copy()
    frame['accepted']=accepted;frame['observed']=observed;frame['agreement']=agreement
    intervals,replicates=foundation_group_intervals(frame,B=1000,seed=0)
    write_json(p['reports']/f'{run_id}_foundation_group_intervals.json',intervals)
    write_table(p['reports']/f'{run_id}_foundation_bootstrap.csv',replicates)
    for metric,detail in intervals['metrics'].items():
        rows[-1][metric+'_ci95_low'],rows[-1][metric+'_ci95_high']=detail['ci95']
    write_json(p['reports']/f'{run_id}_test_calibration.json',calibration_metrics(pr,pred['y'],pred['mask']))
table=pd.DataFrame(rows);write_table(p['reports']/'foundation_test_results.csv',table);display(table)

## 4. Do not rename these results as cancer identification accuracy

In [ ]:
print('This foundation measures recognition and annotation agreement. The flagship needs the independently documented benchmark and support rubric.')
print('Participant-group intervals are saved for each run. Expand to the registered panel without pooling seeds as independent participants; the primary cross-seed PCSI analysis is in notebook 14.')

In [ ]:
MESSAGE = "nb21: foundation calibration, sigmoid best (joint 0.110->0.031, independent 0.225->0.140), selective coverage 0.80, panel disagreement 0.178 independent / 0.502 joint"

import sys, subprocess
r = subprocess.run([sys.executable, "tools/commit_cell.py", MESSAGE],
                   cwd="/content/drive/MyDrive/OncoPlate_Research/oncoplate-research",
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr, file=sys.stderr); print("exit:", r.returncode)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
